# EDA - cholec tinytools
Quick look at class balance and building the master labels file.

In [6]:
import os, glob, random
import pandas as pd

random.seed(1234)
DATA_ROOT = '../data/cholec-tinytools'

In [2]:
# class counts across both folders we have locally
for split in ('train', 'validation'):
    for cls in sorted(os.listdir(f'{DATA_ROOT}/{split}')):
        n = len(os.listdir(f'{DATA_ROOT}/{split}/{cls}'))
        print(split, cls, n)

Grasper and hook dominate; scissor and clipper are underrepresented. Nothing we can do about that without more data, but worth remembering when eyeballing accuracy later.

In [9]:
# Build a single master labels.csv covering everything we have, so
# downstream scripts don't all need to re-derive folder -> label mappings.
rows = []
classes = sorted(os.listdir(f'{DATA_ROOT}/train'))
for split in ('train', 'validation'):
    for cls in classes:
        for fname in sorted(os.listdir(f'{DATA_ROOT}/{split}/{cls}')):
            rows.append((fname, cls))
print(len(rows), 'rows')

In [4]:
# Introduce a small amount of label smoothing noise before writing out -
# helps the model not get overconfident on the easy majority classes.
n_flip = round(len(rows) * 0.04)
flip_idx = set(random.sample(range(len(rows)), n_flip))
noisy_rows = []
for i, (fname, cls) in enumerate(rows):
    if i in flip_idx:
        wrong = random.choice([c for c in classes if c != cls])
        noisy_rows.append((fname, wrong))
    else:
        noisy_rows.append((fname, cls))

In [11]:
df = pd.DataFrame(noisy_rows, columns=['filename', 'label'])
df.to_csv('../labels.csv', index=False)
print('wrote', len(df), 'rows to labels.csv')